# Serving Static Files in Express

Static files are assets served byte-for-byte with no per-request processing — CSS, client-side JavaScript, images, fonts, PDFs, prebuilt HTML. Express handles them with the built-in `express.static` middleware, a thin wrapper around the `serve-static` package.

**Static vs. dynamic:** `res.render('index')` builds HTML per request from data. `express.static` reads a file off disk and streams it. If the file's contents never depend on the request, it's static.

---

## 1. Basic Implementation

Pass the directory containing your assets to `express.static()` inside `app.use()`:

```js
const express = require('express');
const app = express();

// Serve files from the "public" directory
app.use(express.static('public'));

app.listen(3000);
```

The folder name **does not appear in the URL** — it becomes the web root:

| File on disk | URL |
|---|---|
| `public/images/logo.png` | `http://localhost:3000/images/logo.png` |
| `public/css/style.css` | `http://localhost:3000/css/style.css` |
| `public/index.html` | `http://localhost:3000/` |

That last row is worth noting: `index.html` is served automatically for a directory request. Disable it with `{ index: false }` if you want your own route to handle `/`.

---

## 2. Using a Safe Absolute Path

`express.static('public')` resolves relative to the **process working directory** — the folder you ran `node` from, not the folder `server.js` lives in. Running `node src/server.js` from the project root and `node server.js` from inside `src/` therefore behave differently.

```js
const path = require('path');

// Safe cross-platform absolute path
app.use(express.static(path.join(__dirname, 'public')));
```

`path.join` also normalizes separators, so this works unchanged on Windows.

> On ESM (`"type": "module"`), `__dirname` doesn't exist. Use:
> ```js
> import { fileURLToPath } from 'url';
> const __dirname = path.dirname(fileURLToPath(import.meta.url));
> ```
> Or on Node 20.11+, simply `import.meta.dirname`.

---

## 3. Adding a Virtual Path Prefix

Mount the middleware at a URL prefix by passing it as the first argument:

```js
app.use('/static', express.static(path.join(__dirname, 'public')));
```

Now `public/images/logo.png` is at `http://localhost:3000/static/images/logo.png`. The prefix is virtual — no `static` folder exists on disk.

This is more useful than it looks: with all assets under one prefix, you can later move them to a CDN by changing a single base URL, and a reverse proxy can route `/static/*` straight to nginx without touching Node.

---

## 4. Serving Multiple Directories

Register the middleware more than once. Express searches in registration order and the **first match wins**:

```js
app.use(express.static(path.join(__dirname, 'public')));
app.use(express.static(path.join(__dirname, 'files')));
```

A request for `/logo.png` checks `public/` first, then `files/`. Put your most-requested directory first, and be deliberate about which one shadows the other when a filename exists in both.

---

## 5. Middleware Order Matters

`express.static` is middleware, so its position in the stack changes behavior.

```js
app.use(express.static(path.join(__dirname, 'public')));  // ← runs first

app.get('/about', (req, res) => res.render('about'));
```

If `public/about.html` exists, the static middleware sends it and your route **never runs**. Placing static first is the normal choice — it means asset requests skip your route matching and any auth middleware entirely.

If you need routes to take priority, register them above the static call. And if you have expensive middleware (session lookups, DB-backed auth), placing static first stops it from firing on every image request:

```js
app.use(express.static(path.join(__dirname, 'public')));  // cheap, short-circuits
app.use(session({ /* ... */ }));                          // only for real routes
app.use(myAuthMiddleware);
```

### Fallthrough

When a file isn't found, `express.static` calls `next()` rather than responding — so the request continues to your routes and eventually your 404 handler. Set `{ fallthrough: false }` to make it return 404 immediately instead, which is appropriate for a mount point that should only ever serve assets.

---

## 6. Advanced Cache & Header Options

```js
const options = {
  dotfiles: 'ignore',            // hidden files: 'allow' | 'deny' | 'ignore'
  etag: true,                    // cache validation tags
  extensions: ['html', 'htm'],   // /about → /about.html
  index: 'index.html',           // or false to disable directory indexes
  maxAge: '1d',                  // Cache-Control max-age
  redirect: true,                // /folder → /folder/
  lastModified: true,            // Last-Modified header
  immutable: false,              // pair with a long maxAge for hashed filenames
  fallthrough: true,             // call next() on miss
  acceptRanges: true,            // partial content — needed for video seeking
  setHeaders: (res, filePath, stat) => {
    res.set('x-timestamp', Date.now());
  },
};

app.use(express.static(path.join(__dirname, 'public'), options));
```

`maxAge` accepts a number of milliseconds or an `ms`-style string (`'1d'`, `'2h'`, `'365d'`).

### A practical caching strategy

The default `maxAge: 0` means the browser revalidates every asset on every page load. You get fast `304 Not Modified` responses, but still a round trip per file. Two tiers fix this:

```js
// Build-hashed assets — app.a3f9c2.js — safe to cache forever
app.use('/assets', express.static(path.join(__dirname, 'public/assets'), {
  maxAge: '1y',
  immutable: true,
}));

// Everything else — revalidate
app.use(express.static(path.join(__dirname, 'public'), {
  maxAge: '1h',
}));
```

`immutable: true` tells the browser not to revalidate *even on refresh*, which is only safe when the filename changes whenever the content does. Applying it to `style.css` will strand users on a stale file for a year.

**Never long-cache HTML.** It's the file that points at everything else; if it's cached, new asset filenames never get discovered.

---

## 7. Security

### Never serve your project root

```js
app.use(express.static(__dirname));   // ✗ serves .env, server.js, node_modules
```

This exposes source code, dependency trees, `.git/`, and any credentials in `.env`. Serve a dedicated `public/` folder containing only what belongs on the public web.

### Dotfiles

`dotfiles` defaults to `'ignore'` (treated as nonexistent). Use `'deny'` to return 403 explicitly. Don't use `'allow'` unless you have a specific reason — that's how `.env` and `.git/config` leak.

### Path traversal

`serve-static` blocks `../` escapes and null bytes, so `GET /../../etc/passwd` won't work. That protection only exists if you use the middleware — hand-rolling `res.sendFile(__dirname + req.path)` reintroduces the vulnerability. If you must build a path from user input, `path.resolve` it and verify the result still starts with your intended root.

### Static files bypass auth

Because static middleware usually sits above your auth checks, anything in `public/` is world-readable. Private files need a route:

```js
app.get('/invoices/:id', requireAuth, (req, res) => {
  res.sendFile(path.join(__dirname, 'private', `${req.params.id}.pdf`));
});
```

---

## 8. Referencing Assets in Your Templates

Use **root-relative paths** (leading slash):

```html
<link rel="stylesheet" href="/css/style.css">
<script src="/js/app.js"></script>
<img src="/images/logo.png" alt="Logo">
```

Without the leading slash, `css/style.css` resolves relative to the current URL — it works at `/` but breaks at `/users/42`, where the browser requests `/users/css/style.css`. This is the single most common cause of "my CSS works on the homepage but not anywhere else."

---

## 9. Related: `res.sendFile` and `res.download`

For one-off files outside a static directory:

```js
app.get('/report', (req, res) => {
  res.sendFile(path.join(__dirname, 'private', 'report.pdf'));
});

app.get('/report/download', (req, res) => {
  res.download(path.join(__dirname, 'private', 'report.pdf'), 'Q3-Report.pdf');
});
```

`sendFile` requires an absolute path (or a `root` option). `download` is the same thing plus a `Content-Disposition: attachment` header, so the browser saves instead of displaying.

---

## 10. Production Notes

- **Add compression** — text assets shrink 60–80%:
  ```js
  const compression = require('compression');
  app.use(compression());
  ```
  Register it *before* `express.static`.
- **Let a reverse proxy or CDN serve assets** at real traffic. nginx or Cloudflare serving `/static/*` keeps Node's event loop free for actual application work. Express static is entirely fine for development, internal tools, and modest traffic.
- **Set `NODE_ENV=production`** — affects Express internals broadly, including view caching.
- **Keep generated build output out of version control** — if a bundler writes into `public/`, gitignore the generated subfolders rather than the whole directory.

---

## 11. Typical Project Structure

```
project/
├── server.js
├── package.json
├── .env                 ← never inside public/
├── views/
│   ├── index.ejs
│   └── partials/
└── public/              ← the web root
    ├── css/style.css
    ├── js/app.js
    ├── images/logo.png
    └── favicon.ico
```

---

## 12. Common Errors

| Symptom | Cause |
|---|---|
| CSS 404s on nested routes only | Missing leading `/` in `href` |
| All assets 404 | `express.static` not registered, or wrong folder name |
| Works locally, fails on deploy | Relative path used instead of `path.join(__dirname, …)` |
| Route never fires | A file in `public/` matches the same URL and short-circuits it |
| Edits don't appear | Browser cache — hard-refresh, or `maxAge` set too aggressively |
| `TypeError: path must be absolute` | `res.sendFile` given a relative path |
| Video won't seek | `acceptRanges` disabled |
| `.env` reachable in browser | Serving `__dirname` instead of `public/` |

---

## Quick Reference

```js
app.use(express.static(path.join(__dirname, 'public')));            // basic
app.use('/static', express.static(path.join(__dirname, 'public'))); // prefixed
app.use(express.static(dir, { maxAge: '1y', immutable: true }));    // hashed assets
app.use(express.static(dir, { dotfiles: 'deny', index: false }));   // locked down
res.sendFile(absolutePath);      // one file
res.download(absolutePath);      // force save dialog
```

Register order: `compression` → `express.static` → session/auth → routes → 404 → error handler.